# NT-CCTV — Brand Classifier: Fine-tune MobileNetV3-Small

**เป้าหมาย**: เทรน MobileNetV3-Small ให้จำแนกยี่ห้อรถ → export ONNX → ใช้แทน gallery cosine similarity ใน pipeline หลัก

| Step | Cell | เวลา (Colab T4) |
|---|---|---|
| Install + Mount Drive | 1 | ~2 min |
| Config + Dataset | 2 | ~30 s |
| Phase 1 — Head-only training (10 ep) | 3 | ~5 min |
| Phase 2 — Full fine-tune (max 60 ep, early stop) | 4 | ~25–35 min |
| Evaluation + Confusion matrix | 5 | ~1 min |
| ONNX Export + Verify | 6 | ~30 s |
| Pipeline integration snippet | 7 | reference only |

---

### Phase 2 Design
```
ReduceLROnPlateau(patience=5, factor=0.3)  ← ลด LR เมื่อ val_acc plateau
Early stopping patience=15                 ← หยุดเมื่อ ReduceLROnPlateau ช่วยแล้วยังไม่ดีขึ้น
Backbone LR = 1e-4  |  Head LR = 5e-4     ← Differential LR ป้องกัน overwrite ImageNet features
```

---
**ก่อนรัน**: อัพโหลด `Car Brand/` folder ไปที่ `MyDrive/NT-CCTV/Car Brand/` ใน Google Drive ก่อน


In [ ]:
# ═══ CELL 1: Install + Mount Drive ════════════════════════════════════
!pip install -q timm torch torchvision scikit-learn matplotlib seaborn

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — ⚠️ เปิด GPU Runtime ก่อน"}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ═══ CELL 2: Config + Dataset ══════════════════════════════════════════
import json, random
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────────
DATA_DIR    = Path('/content/drive/MyDrive/NT-CCTV/Car Brand')
OUTPUT_DIR  = Path('/content/drive/MyDrive/NT-CCTV/brand_model')
MODEL_NAME  = 'mobilenetv3_small_100'   # Jetson Nano: ~30ms CPU, ~10MB
IMG_SIZE    = 224
BATCH_SIZE  = 64
SEED        = 42
VAL_SPLIT   = 0.20
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f'❌ ไม่พบ {DATA_DIR} — อัพโหลด Car Brand/ ไป Drive ก่อน'

# ── Class names (non-empty folders only) ─────────────────────────────
CLASS_NAMES = sorted([
    d.name for d in DATA_DIR.iterdir()
    if d.is_dir() and any(d.glob('*.jp*g')) or any(d.glob('*.png'))
])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
N_CLASSES = len(CLASS_NAMES)

print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')
json.dump(CLASS_NAMES, open(OUTPUT_DIR / 'class_names.json', 'w'), ensure_ascii=False, indent=2)
print(f'✅ class_names.json saved ({N_CLASSES} classes)')

# ── Gather all image paths ────────────────────────────────────────────
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
all_paths, all_labels = [], []
for cls in CLASS_NAMES:
    for p in (DATA_DIR / cls).iterdir():
        if p.suffix.lower() in IMG_EXTS:
            all_paths.append(p)
            all_labels.append(CLASS_TO_IDX[cls])

# ── Stratified split (every class in both train/val) ─────────────────
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=all_labels,
)

# ── Transforms ───────────────────────────────────────────────────────
# Train: aggressive aug to simulate CCTV conditions
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.70, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.06),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
# Val: clean crop for accurate evaluation
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class BrandDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        return self.transform(img), self.labels[idx]

train_ds = BrandDataset(train_paths, train_labels, train_tf)
val_ds   = BrandDataset(val_paths,   val_labels,   val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

from collections import Counter
cls_counts = Counter(all_labels)
print(f'\nTotal images : {len(all_paths):,}')
print(f'Train        : {len(train_ds):,}')
print(f'Val          : {len(val_ds):,}')
print(f'Batches/epoch: {len(train_loader)}')
print(f'Min/Max per class (train): {min(cls_counts.values())} / {max(cls_counts.values())}')

In [ ]:
# ═══ CELL 3: Phase 1 — Head-only Training (10 epochs) ═════════════════
#  Freeze backbone → train only classifier head → fast convergence
#  เหมาะสำหรับ warm-up ก่อน full fine-tune
import timm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

PHASE1_EPOCHS = 10
PHASE1_LR     = 3e-3

# ── Load pretrained MobileNetV3-Small ─────────────────────────────────
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=N_CLASSES)
model = model.to(DEVICE)

# ── Freeze all except classifier head ─────────────────────────────────
for name, param in model.named_parameters():
    if 'classifier' in name or 'conv_head' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model.parameters())
print(f'Model : {MODEL_NAME}  ({n_total/1e6:.2f}M params)')
print(f'Phase 1 trainable: {n_trainable:,} / {n_total:,}  ({n_trainable/n_total*100:.1f}%)')

# ── Loss: Label smoothing — helps prevent overconfidence on augmented data
criterion  = nn.CrossEntropyLoss(label_smoothing=0.10)
optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=PHASE1_LR, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-5)

best_acc_p1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

SEP = '═' * 62
print(f'\n{SEP}')
print(f'  Phase 1: Head-only  ({PHASE1_EPOCHS} epochs, LR={PHASE1_LR})')
print(SEP)

for epoch in range(1, PHASE1_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # ── Validate ────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total_n = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_loss += criterion(out, labels).item()
            correct  += (out.argmax(1) == labels).sum().item()
            total_n  += labels.size(0)

    acc = correct / total_n * 100
    tl  = running_loss / len(train_loader)
    vl  = val_loss     / len(val_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(acc)
    scheduler.step()

    mark = ' ← best' if acc > best_acc_p1 else ''
    print(f'  Ep {epoch:2d}/{PHASE1_EPOCHS}  train_loss={tl:.4f}  val_loss={vl:.4f}  val_acc={acc:6.2f}%{mark}')

    if acc > best_acc_p1:
        best_acc_p1 = acc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'class_names': CLASS_NAMES, 'model_name': MODEL_NAME},
                   str(OUTPUT_DIR / 'brand_phase1_best.pt'))

print(f'{SEP}')
print(f'  Phase 1 best val_acc: {best_acc_p1:.2f}%  → brand_phase1_best.pt')

In [ ]:
# ═══ CELL 4: Phase 2 — Full Fine-tune (max 60 epochs) ═════════════════
#
#  การออกแบบ:
#   - ReduceLROnPlateau(patience=5, factor=0.3):
#       ถ้า val_acc ไม่ขึ้น 5 epochs → ลด LR × 0.3  (ช่วย escape plateau)
#   - Early stopping patience=15:
#       ถ้ายังไม่ดีขึ้นอีก 15 epochs หลัง LR ลดแล้ว → หยุด
#   - Backbone LR × 0.2 (1e-4), Head LR = 5e-4:
#       ไม่ทำลาย ImageNet features ที่ pretrained มา
#
#  ทำไม ReduceLROnPlateau ดีกว่า CosineAnnealingLR ในกรณีนี้:
#   CosineAnnealingLR มี plateau ช่วงกลาง → early stopping อาจ trigger ก่อนกราฟ
#   ลงต่ำสุด ReduceLROnPlateau ลด LR เฉพาะเมื่อจำเป็น → ทำงานร่วมกับ
#   early stopping ได้ถูกต้อง
# ──────────────────────────────────────────────────────────────────────
import torch
from torch.optim import AdamW

PHASE2_EPOCHS = 60    # max epochs — early stopping จะหยุดเองถ้า converge แล้ว
PHASE2_LR     = 5e-4
PATIENCE      = 15    # early stopping — ให้ ReduceLROnPlateau มีโอกาสช่วยก่อน

# ── โหลด best Phase 1 weights ──────────────────────────────────────────
ckpt = torch.load(str(OUTPUT_DIR / 'brand_phase1_best.pt'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded Phase 1 checkpoint (epoch={ckpt["epoch"]}, best_acc={best_acc_p1:.2f}%)')

# ── Unfreeze ทุก layer ─────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = True

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable: {n_trainable:,} (all layers)')

# ── Differential LR: backbone เล็กกว่า head 5× ───────────────────────
optimizer = AdamW([
    {'params': [p for n, p in model.named_parameters()
                if 'classifier' not in n and 'conv_head' not in n],
     'lr': PHASE2_LR * 0.2},    # backbone: 1e-4
    {'params': [p for n, p in model.named_parameters()
                if 'classifier' in n or 'conv_head' in n],
     'lr': PHASE2_LR},           # head: 5e-4
], weight_decay=1e-4)

# ReduceLROnPlateau: track val_acc (mode='max')
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.3, patience=5, min_lr=1e-7, verbose=True
)

best_acc_p2    = 0.0
patience_count = 0

SEP = '═' * 68
print(f'\n{SEP}')
print(f'  Phase 2: Full fine-tune  (max {PHASE2_EPOCHS} ep | head_LR={PHASE2_LR} | backbone_LR={PHASE2_LR*0.2})')
print(f'  ReduceLROnPlateau: factor=0.3  patience=5  |  early-stop patience={PATIENCE}')
print(SEP)

for epoch in range(1, PHASE2_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # ── Validate ────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total_n = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_loss += criterion(out, labels).item()
            correct  += (out.argmax(1) == labels).sum().item()
            total_n  += labels.size(0)

    acc = correct / total_n * 100
    tl  = running_loss / len(train_loader)
    vl  = val_loss     / len(val_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(acc)

    # ReduceLROnPlateau ต้องการ metric (val_acc) ไม่ใช่แค่ step()
    scheduler.step(acc)
    cur_lr = optimizer.param_groups[-1]['lr']   # head LR

    if acc > best_acc_p2:
        best_acc_p2    = acc
        patience_count = 0
        torch.save({'epoch': epoch + PHASE1_EPOCHS, 'model_state': model.state_dict(),
                    'class_names': CLASS_NAMES, 'model_name': MODEL_NAME},
                   str(OUTPUT_DIR / 'brand_best.pt'))
        mark = ' ← best ✓'
    else:
        patience_count += 1
        mark = f' (patience {patience_count}/{PATIENCE})'

    print(f'  Ep {epoch:2d}/{PHASE2_EPOCHS}  train={tl:.4f}  val={vl:.4f}  acc={acc:6.2f}%  lr={cur_lr:.1e}{mark}')

    if patience_count >= PATIENCE:
        print(f'\n  ⏹  Early stopping (no improvement for {PATIENCE} epochs)')
        break

print(f'{SEP}')
print(f'  Phase 2 best val_acc: {best_acc_p2:.2f}%  → brand_best.pt')


In [ ]:
# ═══ CELL 5: Evaluation + Confusion Matrix ═════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 8
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# ── โหลด best model ────────────────────────────────────────────────────
ckpt = torch.load(str(OUTPUT_DIR / 'brand_best.pt'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]}, best_acc={best_acc_p2:.2f}%)')

# ── Collect predictions ────────────────────────────────────────────────
all_preds, all_true, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1)
        all_preds.extend(probs.argmax(1).cpu().tolist())
        all_true.extend(labels.tolist())
        all_probs.extend(probs.max(1).values.cpu().tolist())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

top1_acc = (all_preds == all_true).mean() * 100
print(f'\n  Top-1 Accuracy : {top1_acc:.2f}%')
print(f'  Mean confidence: {np.mean(all_probs):.3f}')

# ── Per-class report ───────────────────────────────────────────────────
print('\n' + '─' * 60)
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=3))

# ── Training curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['train_loss'], label='Train loss')
axes[0].plot(history['val_loss'],   label='Val loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history['val_acc'])
axes[1].axhline(top1_acc, color='red', linestyle='--', label=f'Best {top1_acc:.1f}%')
axes[1].set_title('Val Accuracy'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=120)
plt.show()

# ── Confusion matrix ───────────────────────────────────────────────────
cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100  # row-normalized %

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, vmin=0, vmax=100, cbar_kws={'label': 'Recall (%)'})
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {MODEL_NAME}  (val_acc={top1_acc:.1f}%)')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=120)
plt.show()

# ── Classes with lowest accuracy ──────────────────────────────────────
per_class_acc = np.diag(cm) / cm.sum(axis=1) * 100
low_acc = sorted(zip(CLASS_NAMES, per_class_acc), key=lambda x: x[1])
print('\n  Lowest per-class accuracy:')
for cls, acc in low_acc[:8]:
    bar = '█' * int(acc / 5)
    print(f'  {cls:<20} {acc:5.1f}%  {bar}')

In [ ]:
# ═══ CELL 6: ONNX Export + Verify ══════════════════════════════════════
#  Export พร้อม softmax output → ใช้งานใน pipeline ด้วย onnxruntime
#  Output shape: [batch, N_CLASSES] probabilities
import torch
import numpy as np

ONNX_PATH = OUTPUT_DIR / 'brand_classifier.onnx'

# ── โหลด best model ────────────────────────────────────────────────────
ckpt = torch.load(str(OUTPUT_DIR / 'brand_best.pt'), map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model.eval().cpu()

# ── Wrap with softmax for direct probability output ───────────────────
class BrandClassifierWithSoftmax(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x):
        return torch.softmax(self.base(x), dim=1)

export_model = BrandClassifierWithSoftmax(model)
export_model.eval()

dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    torch.onnx.export(
        export_model,
        dummy,
        str(ONNX_PATH),
        input_names=['image'],
        output_names=['probs'],
        dynamic_axes={'image': {0: 'batch'}, 'probs': {0: 'batch'}},
        opset_version=12,
        do_constant_folding=True,
    )

size_mb = ONNX_PATH.stat().st_size / 1e6
print(f'✅  {ONNX_PATH.name}  ({size_mb:.2f} MB)')

# ── Verify ONNX output ──────────────────────────────────────────────────
import onnxruntime as ort
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])

test_img = dummy.numpy()
probs_ort = sess.run(None, {'image': test_img})[0][0]  # shape: (N_CLASSES,)

assert abs(probs_ort.sum() - 1.0) < 1e-4, 'Probabilities should sum to 1'
top3_idx  = probs_ort.argsort()[::-1][:3]
top3      = [(CLASS_NAMES[i], float(probs_ort[i])) for i in top3_idx]

print(f'\nVerification (dummy input):')
for brand, prob in top3:
    print(f'  {brand:<20} {prob:.4f}')
print(f'  Sum of probs: {probs_ort.sum():.6f} ✓')
print(f'\n  Output: [{ONNX_PATH.name}] → [{OUTPUT_DIR / "class_names.json"}]')
print(f'  Copy both files ไปไว้ใน NT-CCTV/models/brand/')

# ── Benchmark inference time (CPU, Nano-like) ──────────────────────────
import time
N_BENCH = 100
t0 = time.perf_counter()
for _ in range(N_BENCH):
    sess.run(None, {'image': test_img})
ms_per = (time.perf_counter() - t0) / N_BENCH * 1000
print(f'  Inference (CPU, Colab): {ms_per:.1f} ms/crop  (Jetson Nano ORT ~15-30ms)')

# Cell 7 — Pipeline Integration

คัดลอก code ด้านล่างไปแทนที่ **Cell 3 (Brand Similarity)** ใน `NT CCTV Stream Test3 copy.ipynb`

ไฟล์ที่ต้องการ (copy ไปไว้ใน Colab working dir):
- `brand_classifier.onnx`
- `class_names.json`

In [ ]:
# ═══ CELL 7: Pipeline Integration Snippet ══════════════════════════════
#  ← แทนที่ Cell 3 (Brand Similarity) ในไฟล์ pipeline หลัก
#  ไม่ต้องเปลี่ยน Cell 4 (Main Pipeline) เพราะยังเรียก infer_brand_prior() เหมือนเดิม
import json
import threading
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import onnxruntime as ort

# ── Paths (adjust ถ้า save ไว้ที่อื่น) ────────────────────────────────
BRAND_ONNX_PATH   = Path('brand_classifier.onnx')
BRAND_NAMES_PATH  = Path('class_names.json')

assert BRAND_ONNX_PATH.exists(),  f'❌ {BRAND_ONNX_PATH} not found'
assert BRAND_NAMES_PATH.exists(), f'❌ {BRAND_NAMES_PATH} not found'

# ── Load classifier ────────────────────────────────────────────────────
_BRAND_NAMES: list[str] = json.loads(BRAND_NAMES_PATH.read_text())
_brand_sess = ort.InferenceSession(
    str(BRAND_ONNX_PATH),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
_brand_lock = threading.Lock()   # serialize CPU inference across 5 camera threads

# ── Preprocessing (same as training val_tf) ───────────────────────────
_brand_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

CONF_THRESHOLD = 0.35   # ต่ำกว่านี้ → 'Unknown'

def infer_brand_prior(cls_name, color=None, _crop_bgr=None):
    """Drop-in replacement for the original gallery-based infer_brand_prior().

    Returns: (brand: str, confidence: float, source: str)
    source = 'classifier' | 'unknown'

    Thread-safe: _brand_lock serializes ORT CPU inference across 5 threads.
    """
    if _crop_bgr is None or _crop_bgr.size == 0:
        return 'Unknown', 0.0, 'unknown'

    # BGR → RGB → PIL → tensor → numpy
    rgb = cv2.cvtColor(_crop_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    x   = _brand_tf(pil).unsqueeze(0).numpy()  # (1, 3, 224, 224)

    with _brand_lock:
        probs = _brand_sess.run(None, {'image': x})[0][0]  # (N_CLASSES,)

    idx  = int(probs.argmax())
    conf = float(probs[idx])

    if conf < CONF_THRESHOLD:
        return 'Unknown', conf, 'unknown'

    return _BRAND_NAMES[idx], round(conf, 3), 'classifier'


print(f'✅  Brand classifier ready')
print(f'   Model  : {BRAND_ONNX_PATH.name}')
print(f'   Classes: {len(_BRAND_NAMES)}')
print(f'   Brands : {_BRAND_NAMES}')
print(f'   Conf threshold: {CONF_THRESHOLD}')

# ── Quick smoke test ───────────────────────────────────────────────────
test_crop = np.random.randint(0, 255, (120, 160, 3), dtype=np.uint8)
brand, conf, src = infer_brand_prior('Car', _crop_bgr=test_crop)
print(f'\n  Smoke test → brand={brand}  conf={conf:.3f}  src={src}')
print('  (random noise input — conf ต่ำปกติ)')